# How Craig Wright Fooled Gavin Andresen

**The anatomy of a fake Satoshi proof — and what it teaches us about ECDSA**

[← Back to ECC Teachable Scheme](./00-ecc-teachable-scheme.ipynb) | [The Nonsense Signature (2018) →](./02-ecc-nonsense-signature.ipynb)

---

## The Setup

In May 2016, Craig Wright claimed to be Satoshi Nakamoto. He published a
"proof" on his blog and gave a private demonstration to Gavin Andresen
(then the lead maintainer of Bitcoin Core), who walked away convinced.

Both proofs were fake. But they teach us something deep about what
signatures actually prove — and what they don't.

```
What a signature PROVES:       What a signature DOES NOT prove:
─────────────────────────      ──────────────────────────────────
This (r,s) is valid for        WHO produced it
this message hash z            WHEN it was produced
under this public key P        That the signer is in front of you
```

**The fundamental lesson:** Verification tells you a signature is *valid*.
It does not tell you the person showing it to you *created* it.

---

## Part 1: The Blog Post Trick (The Sleight of Hand)

Wright published a blog post claiming he signed a text from Jean-Paul Sartre
using Satoshi's private key. Here's how the trick actually worked.

### The Honest Way to Prove Identity

```
1. Someone gives you a FRESH, UNPREDICTABLE message
   → "Sign this: 'The weather in London on May 2, 2016 is rainy'"

2. You sign it with your private key
   → sig = ECDSA_sign(message, private_key)

3. They verify using the known public key
   → ECDSA_verify(message, sig, satoshi_pubkey) == True

4. Since only the private key holder can produce valid signatures
   for fresh messages, this proves identity ✓
```

### What Wright Actually Did

```
1. Take a real Satoshi transaction from 2009 (public on the blockchain)
   → tx 828ef3b0... (Satoshi sending 10 BTC in January 2009)

2. Extract the signature from that transaction
   → This signature is already valid under Satoshi's key
   → ANYONE can read it — it's public data

3. Present it as if he just created it for a Sartre text
   → "Look, I signed this Sartre file with Satoshi's key!"

4. The verification passes — because it's a REAL signature
   → But he didn't create it. He copied it.
```

### The SHA256 Bridge

The clever part was making the old transaction signature *look like* a
signature of a new file. Here's how:

```
Bitcoin signs transactions like this:
  z = SHA256(SHA256(modified_transaction))
  sig = ECDSA_sign(z, private_key)

OpenSSL verifies files like this:
  z = SHA256(file_contents)
  ECDSA_verify(z, sig, pubkey)

Wright's trick:
  The "Sartre file" WAS the SHA256(modified_transaction)
  So OpenSSL computes: SHA256("Sartre") = SHA256(SHA256(modtx))
  Which is exactly z from the original Bitcoin transaction!

  Same z → same valid signature → verification passes
```

In [1]:
import hashlib

# The real hash from Satoshi's 2009 transaction (tx 828ef3b0...)
# This is SHA256 of the "modified transaction" (signature_form)
modtx_hash_hex = "479f9dff0155c045da78402177855fdb4f0f396dc0d2c24f7376dd56e2e68b05"
modtx_hash = bytes.fromhex(modtx_hash_hex)

# Bitcoin computes z as: SHA256(SHA256(modified_transaction))
# Wright's "Sartre file" contained the raw bytes of SHA256(modified_transaction)
# So when OpenSSL does SHA256(Sartre_file), it gets SHA256(SHA256(modtx)) = z

sartre_is_modtx_hash = modtx_hash  # The "Sartre file" is literally this hash

# What OpenSSL computes when "verifying" the Sartre file:
z_openssl = hashlib.sha256(sartre_is_modtx_hash).hexdigest()

# What Bitcoin computed for the original 2009 transaction:
z_bitcoin = hashlib.sha256(modtx_hash).hexdigest()

print("=== The SHA256 Bridge ===")
print(f"\n'Sartre file' contents (hex):")
print(f"  {modtx_hash_hex}")
print(f"\nOpenSSL verification computes z = SHA256('Sartre file'):")
print(f"  {z_openssl}")
print(f"\nBitcoin's original z = SHA256(SHA256(modified_tx)):")
print(f"  {z_bitcoin}")
print(f"\nSame z? {z_openssl == z_bitcoin}")
print(f"\nSince z is the same, the original transaction signature")
print(f"is also valid as a 'signature of the Sartre file'.")
print(f"But Wright never touched a private key. He just copied.")

=== The SHA256 Bridge ===

'Sartre file' contents (hex):
  479f9dff0155c045da78402177855fdb4f0f396dc0d2c24f7376dd56e2e68b05

OpenSSL verification computes z = SHA256('Sartre file'):
  3ec9cbc0d1aa849c16a1b276b246e057e7232b21926e428cc09b692c14336f44

Bitcoin's original z = SHA256(SHA256(modified_tx)):
  3ec9cbc0d1aa849c16a1b276b246e057e7232b21926e428cc09b692c14336f44

Same z? True

Since z is the same, the original transaction signature
is also valid as a 'signature of the Sartre file'.
But Wright never touched a private key. He just copied.


### Let's Reproduce the Trick

We'll simulate exactly what Wright did, using our own keys instead of Satoshi's.
This proves **anyone** can do this — no private key needed.

In [2]:
import secrets
from typing import Optional

# ── secp256k1 primitives (from teachable scheme) ──
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

class Point:
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    def is_infinity(self):
        return self.x is None or self.y is None
    def copy(self):
        return Point(self.x, self.y)
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity(): return True
        return self.x == other.x and self.y == other.y

G = Point(SECP_GX, SECP_GY)

def mod_inverse(a, p):
    return pow(a, p - 2, p)

def point_add(p1, p2):
    if p1.is_infinity(): return p2.copy()
    if p2.is_infinity(): return p1.copy()
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0: return Point()
        lam = (3 * p1.x * p1.x * mod_inverse(2 * p1.y, SECP_P)) % SECP_P
    else:
        lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def scalar_mult(k, p):
    if k == 0 or p.is_infinity(): return Point()
    k = k % SECP_N
    if k == 0: return Point()
    result = Point()
    addend = p.copy()
    while k > 0:
        if k & 1: result = point_add(result, addend)
        addend = point_add(addend, addend)
        k >>= 1
    return result

def ecdsa_sign(z: int, d: int) -> tuple:
    while True:
        k = secrets.randbelow(SECP_N - 1) + 1
        R = scalar_mult(k, G)
        r = R.x % SECP_N
        if r == 0: continue
        s = (pow(k, SECP_N - 2, SECP_N) * (z + r * d)) % SECP_N
        if s == 0: continue
        return (r, s)

def ecdsa_verify(z: int, sig: tuple, pub: Point) -> bool:
    r, s = sig
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u1 = (z * s_inv) % SECP_N
    u2 = (r * s_inv) % SECP_N
    R = point_add(scalar_mult(u1, G), scalar_mult(u2, pub))
    return R.x % SECP_N == r

print("Crypto primitives loaded. ✓")

Crypto primitives loaded. ✓


In [3]:
# ═══════════════════════════════════════════════════════════════
#  SIMULATION: The Wright Trick
# ═══════════════════════════════════════════════════════════════

# Step 1: "Satoshi" creates a real transaction and signs it
print("STEP 1: The real Satoshi signs a transaction in 2009")
print("=" * 55)

satoshi_private = secrets.randbelow(SECP_N - 1) + 1
satoshi_public = scalar_mult(satoshi_private, G)

# Simulate a transaction being signed
# Bitcoin: z = SHA256(SHA256(modified_tx))
fake_transaction = b"tx: send 10 BTC from Satoshi to Hal Finney, Jan 2009"
inner_hash = hashlib.sha256(fake_transaction).digest()  # SHA256(modtx)
z_original = int.from_bytes(hashlib.sha256(inner_hash).digest(), 'big')  # SHA256(SHA256(modtx))

original_sig = ecdsa_sign(z_original, satoshi_private)
r_orig, s_orig = original_sig

print(f"  Satoshi's public key: ({hex(satoshi_public.x)[:18]}...)")
print(f"  Transaction signed. Signature (r, s) recorded on blockchain.")
print(f"  r = {hex(r_orig)[:18]}...")
print(f"  s = {hex(s_orig)[:18]}...")
print(f"  z (message hash) = {hex(z_original)[:18]}...")
print(f"\n  This signature + transaction are PUBLIC on the blockchain.")
print(f"  Anyone in the world can read them.")

STEP 1: The real Satoshi signs a transaction in 2009
  Satoshi's public key: (0xcb9de64b01eacedf...)
  Transaction signed. Signature (r, s) recorded on blockchain.
  r = 0xdbfcdb0a19dce3d8...
  s = 0x9c0028cf9926043e...
  z (message hash) = 0xb076a7a1a8a42399...

  This signature + transaction are PUBLIC on the blockchain.
  Anyone in the world can read them.


In [4]:
# Step 2: Craig Wright (the attacker) reads the blockchain
print("\nSTEP 2: The attacker reads the blockchain (2016)")
print("=" * 55)

# The attacker does NOT have Satoshi's private key.
# But he CAN read the transaction and its signature from the public blockchain.
stolen_sig = original_sig  # Copied from blockchain — no private key needed
stolen_inner_hash = inner_hash  # SHA256(modified_tx) — derivable from public tx data

print(f"  Attacker reads the 2009 transaction from the blockchain.")
print(f"  Copies the signature: (r, s)")
print(f"  Computes SHA256(modified_tx) from the public transaction data.")
print(f"  inner_hash = {stolen_inner_hash.hex()[:32]}...")


STEP 2: The attacker reads the blockchain (2016)
  Attacker reads the 2009 transaction from the blockchain.
  Copies the signature: (r, s)
  Computes SHA256(modified_tx) from the public transaction data.
  inner_hash = 8127117ae34ac9ba0b9b7bb966540669...


In [5]:
# Step 3: The trick — make the old signature look like a new one
print("\nSTEP 3: The trick")
print("=" * 55)

# The attacker creates a file whose contents are SHA256(modified_tx)
# He calls it "Sartre" and claims it's a Jean-Paul Sartre text
sartre_file = stolen_inner_hash  # This IS the trick

print(f"  Attacker creates a file called 'Sartre'.")
print(f"  Claims it contains a Sartre text (shows only first 14% on screen).")
print(f"  Actual contents: raw bytes of SHA256(modified_tx)")
print(f"")
print(f"  Now the attacker says:")
print(f'  "I signed this Sartre file with Satoshi\'s key. Verify it!"')
print(f"")
print(f"  He tells you to verify using OpenSSL:")
print(f"    openssl dgst -verify pub.pem -signature sig.der Sartre")
print(f"")
print(f"  OpenSSL computes: z = SHA256(file_contents)")
print(f"                      = SHA256(SHA256(modified_tx))")
print(f"                      = the EXACT SAME z from the 2009 transaction!")


STEP 3: The trick
  Attacker creates a file called 'Sartre'.
  Claims it contains a Sartre text (shows only first 14% on screen).
  Actual contents: raw bytes of SHA256(modified_tx)

  Now the attacker says:
  "I signed this Sartre file with Satoshi's key. Verify it!"

  He tells you to verify using OpenSSL:
    openssl dgst -verify pub.pem -signature sig.der Sartre

  OpenSSL computes: z = SHA256(file_contents)
                      = SHA256(SHA256(modified_tx))
                      = the EXACT SAME z from the 2009 transaction!


In [6]:
# Step 4: Verification — it passes, but proves nothing
print("\nSTEP 4: Verification")
print("=" * 55)

# What OpenSSL does: z = SHA256(sartre_file)
z_from_sartre = int.from_bytes(hashlib.sha256(sartre_file).digest(), 'big')

print(f"  z from original 2009 tx: {hex(z_original)[:24]}...")
print(f"  z from 'Sartre' file:    {hex(z_from_sartre)[:24]}...")
print(f"  Same z? {z_original == z_from_sartre}")
print(f"")

# Verify the stolen signature against the "Sartre" z
valid = ecdsa_verify(z_from_sartre, stolen_sig, satoshi_public)
print(f"  ECDSA verify(sartre_z, stolen_sig, satoshi_pubkey) = {valid}")
print(f"")
print(f"  The verification PASSES. ✓")
print(f"  But the attacker NEVER touched the private key.")
print(f"  He just copied a signature from the blockchain")
print(f"  and wrapped it in a misleading presentation.")


STEP 4: Verification
  z from original 2009 tx: 0xb076a7a1a8a42399468a1f...
  z from 'Sartre' file:    0xb076a7a1a8a42399468a1f...
  Same z? True

  ECDSA verify(sartre_z, stolen_sig, satoshi_pubkey) = True

  The verification PASSES. ✓
  But the attacker NEVER touched the private key.
  He just copied a signature from the blockchain
  and wrapped it in a misleading presentation.


In [7]:
# Step 5: How to ACTUALLY prove identity (the right way)
print("\nSTEP 5: How REAL proof of identity works")
print("=" * 55)

# The verifier chooses a FRESH, UNPREDICTABLE challenge message
challenge = f"Sign this: random nonce {secrets.token_hex(16)}, date 2016-05-02"
z_challenge = int.from_bytes(hashlib.sha256(challenge.encode()).digest(), 'big')

print(f"  Verifier creates a fresh challenge:")
print(f'  "{challenge}"')
print(f"")

# Only the real Satoshi can sign this
real_sig = ecdsa_sign(z_challenge, satoshi_private)
valid_real = ecdsa_verify(z_challenge, real_sig, satoshi_public)
print(f"  Real Satoshi signs it: valid = {valid_real} ✓")

# The attacker CANNOT sign it — he doesn't have the private key
attacker_private = secrets.randbelow(SECP_N - 1) + 1  # Wrong key
fake_sig = ecdsa_sign(z_challenge, attacker_private)
valid_fake = ecdsa_verify(z_challenge, fake_sig, satoshi_public)
print(f"  Attacker tries:       valid = {valid_fake} ✗")

# Can the attacker reuse the old signature? No — different z
valid_replay = ecdsa_verify(z_challenge, stolen_sig, satoshi_public)
print(f"  Replay old signature: valid = {valid_replay} ✗")
print(f"")
print(f"  A fresh challenge defeats the trick because:")
print(f"  - The attacker can't produce (r,s) for a new z without the key")
print(f"  - Old signatures don't verify against new message hashes")
print(f"  - The verifier controls the message, not the prover")


STEP 5: How REAL proof of identity works
  Verifier creates a fresh challenge:
  "Sign this: random nonce c237da332046414bf41d8aa1133d9fe3, date 2016-05-02"

  Real Satoshi signs it: valid = True ✓
  Attacker tries:       valid = False ✗
  Replay old signature: valid = False ✗

  A fresh challenge defeats the trick because:
  - The attacker can't produce (r,s) for a new z without the key
  - Old signatures don't verify against new message hashes
  - The verifier controls the message, not the prover


---

## Part 2: The Private Demonstration (Fooling Andresen)

The blog trick was caught within hours by the Bitcoin community. But the
private demonstration fooled Gavin Andresen for days.

### What Happened

Wright flew Andresen to London and performed a "live" signing in a hotel room.
Andresen brought a USB stick with Electrum (a Bitcoin wallet with signature
verification). Wright used a "brand new, factory-sealed laptop."

### How It Was (Probably) Faked

```
Attack vector: modified verification software

Option A — Tampered USB:
  Wright's nearby computer silently wrote modified Electrum
  to Andresen's USB stick. The modified version shows "Valid"
  for ANY signature. That's a one-line code change:

    # Original Electrum code:
    if signature_valid:
        show("Signature is VALID")
    else:
        show("Signature is INVALID")

    # Modified (Wright's version):
    if True:  # ← always valid
        show("Signature is VALID")
    else:
        show("Signature is INVALID")

Option B — Pre-loaded laptop:
  The "factory-sealed" laptop already had modified software.
  Factory seals are trivially reapplied.

Option C — Controlled environment:
  The entire demo was in Wright's hotel room.
  He controlled every variable.
```

### Why Andresen Was Vulnerable

Andresen is a brilliant programmer. But the trick wasn't cryptographic —
it was **social engineering** combined with **environment control**.

```
What Andresen verified:     What he SHOULD have verified:
─────────────────────────   ──────────────────────────────────
Software said "Valid"        Was the software actually Electrum?
Laptop looked new            Was the laptop actually clean?
USB was "his"                Was the USB modified when inserted?
Wright seemed confident      Was the math actually happening?
```

---

## Part 3: The ECDSA Lesson

### Signatures are proofs of KNOWLEDGE, not proofs of IDENTITY

ECDSA proves: *"someone who knows private key $d$ produced this $(r, s)$
for message hash $z$."*

It does NOT prove: *"the person showing you this signature is that someone."*

### The verification equation doesn't care about time

$$R' = \frac{z}{s} \times G + \frac{r}{s} \times P$$

This equation works identically whether the signature was:
- Created 5 seconds ago by the person in front of you
- Created 7 years ago by someone else and copied from a public ledger

### The defense: Challenge-Response

```
Correct identity proof protocol:

  Verifier                         Prover
  ────────                         ──────
  1. Generate random nonce
  2. Send challenge message  ──→
                                   3. Sign with private key
                             ←──   4. Send signature
  5. Verify signature
  6. Accept ONLY if:
     - Signature is valid
     - Message matches YOUR challenge
     - Challenge was unpredictable
     - YOU controlled the software
```

### Properties that defeated the trick

| Property | Why it matters |
|----------|---------------|
| **Freshness** | The challenge must be new — old signatures can't be replayed |
| **Unpredictability** | The prover can't pre-compute a valid signature |
| **Verifier control** | The verifier must control the software and environment |
| **Transparency** | The full message and signature must be inspectable |

In [8]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE: Be the Verifier
# ═══════════════════════════════════════════════════════════════
#
#  Below, someone claims to be "Satoshi" and gives you a public key
#  and a signature. Your job: determine if this is a legitimate
#  proof or a Wright-style replay.
#
#  Hint: Check if the signature matches YOUR challenge,
#  not some message the "prover" chose.

# The "Satoshi" public key (this is the key we're testing)
test_priv = secrets.randbelow(SECP_N - 1) + 1
test_pub = scalar_mult(test_priv, G)

# An old signature from the "blockchain" (anyone can see this)
old_message = b"Send 50 BTC to pizza guy"
old_z = int.from_bytes(hashlib.sha256(old_message).digest(), 'big')
old_sig = ecdsa_sign(old_z, test_priv)

# YOUR fresh challenge
your_challenge = f"Prove identity: nonce={secrets.token_hex(32)}"
your_z = int.from_bytes(hashlib.sha256(your_challenge.encode()).digest(), 'big')

# The prover gives you a signature. But which message was it for?
prover_sig = old_sig  # ← The prover is replaying the old signature!

# YOUR verification:
valid_for_old = ecdsa_verify(old_z, prover_sig, test_pub)
valid_for_challenge = ecdsa_verify(your_z, prover_sig, test_pub)

print("=== Verifier's Analysis ===")
print(f"\nYour challenge: {your_challenge[:50]}...")
print(f"\nDoes the signature verify?")
print(f"  Against old message:    {valid_for_old}  (but this proves nothing!)")
print(f"  Against YOUR challenge: {valid_for_challenge}  (this is what matters)")
print(f"\nVerdict: {'FRAUD — replayed signature!' if not valid_for_challenge else 'LEGITIMATE'}")
print(f"\nThe prover gave you a valid signature — but not for YOUR message.")
print(f"Classic Wright trick.")

=== Verifier's Analysis ===

Your challenge: Prove identity: nonce=551dbc6264033dff590e2aac8411...

Does the signature verify?
  Against old message:    True  (but this proves nothing!)
  Against YOUR challenge: False  (this is what matters)

Verdict: FRAUD — replayed signature!

The prover gave you a valid signature — but not for YOUR message.
Classic Wright trick.


---

## Timeline

```
2009-01     Satoshi mines early blocks, signs transactions
            (signatures are on the blockchain forever)

2016-04     Wright contacts Andresen, BBC, The Economist
2016-05-02  Wright publishes blog "proof" (Sartre file trick)
2016-05-02  Dan Kaminsky, ryanc, Reddit expose the trick within HOURS
2016-05-02  Andresen posts "I believe Craig Wright is Satoshi"
2016-05-05  Wright promises more proof, then deletes blog post
2016-05-06  Andresen's commit access to Bitcoin Core revoked

2024-03     UK court rules: Wright is NOT Satoshi (COPA v Wright)
```

## Key Takeaways

1. **Verification ≠ Authentication.** A valid signature proves the math works, not who did the math.

2. **Public signatures can be replayed.** Every Bitcoin signature is public. Anyone can present one as "theirs."

3. **Challenge-response is essential.** The verifier must control the message being signed.

4. **Control the environment.** If the prover controls the software, the demo is worthless.

5. **Extraordinary claims need transparent proof.** A real Satoshi would simply sign a fresh message and publish it for everyone to verify independently.

---

*Sources:*
- *[Recreating Craig Wright's Sartre File](https://rya.nc/sartre.html) — ryanc*
- *[Satoshi: how Craig Wright's deception worked](https://blog.erratasec.com/2016/05/satoshi-how-craig-wrights-deception.html) — Robert Graham, Errata Security*
- *[How Gavin Andresen was duped](https://www.cryptologie.net/article/350/how-gavin-andresen-was-duped-into-believing-wright-is-satoshi/) — David Wong*
- *COPA v Wright [2024] EWHC 1198*